In [107]:
import numpy as np
import pandas as pd

train = pd.read_csv(
    "data/train.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
test = pd.read_csv(
    "data/test.csv",
    parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"],
)
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])

print(train.shape, test.shape, events.shape)
print("доля ботов в train:", train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


## Осмотреться

Прежде чем считать агрегаты, стоит посмотреть на данные: какие типы событий бывают, что
лежит в `platform` и `user_agent`, где пропуски, всё ли уникально.

In [108]:
print(events.event_name.value_counts(), "\n")
print(events.platform.value_counts(), "\n")
print("пропуски по колонкам:")
print(events.isna().mean().round(3))

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64


## Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [109]:
def events_in_window(events, meta, with_target=False):
    cols = ["cookie_id", "window_start_ts", "window_end_ts"]
    if with_target:
        cols.append("target")
    ev = events.merge(meta[cols], on="cookie_id")
    ev = ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]
    return ev


ev_tr = events_in_window(events, train, with_target=True)
ev_te = events_in_window(events, test)  # в test таргета нет
print(len(ev_tr), len(ev_te))

198436 89690


## Ход решения

**Первый подход.** Собрал базовый набор из 13–15 признаков (активность, темп, платформа, глубина поиска, pointer-пропуски, UA-паттерны). RandomForest дал P@R0.7 ≈ 0.2 на test — выше baseline (0.09), но мало для антифрода.

**Расширение.** Использовал всё, что даёт сырая схема — признаки из `event_ts`, `eid`, `platform`, `user_agent`, `item_*`, `search_query`, `search_page`, `pointer_*`, плюс производные отношения. Итого 50+ признаков.

**Отбор по AUC.** Прогнал AUC-скрининг каждого признака. Всё, что на уровне случайности (AUC ~ 0.5), выбросил: `has_ios`, `has_desktop`, `depth_max`, `depth_median`, `dt_mean`, `dt_std`, версии браузеров и ОС.

**Пропуски.** Счётчики и флаги (`n_events`, `n_desktop_events`, `has_search`) → `fillna(0)` — нет событий, значит 0. Остальные (`dt_median`, `depth_mean`, `pointer_missing_share`, `eid_std`) → медиана train, потому что 0 здесь смешивает разные ситуации.

**Optuna.** Подбор гиперпараметров с **time-based CV** внутри objective — train всегда раньше validation, как в реальной задаче. Метрика оптимизации — `precision_at_recall`, учёт дисбаланса через `class_weight` .

## Сбор признаков


In [110]:
import numpy as np
import pandas as pd


def basic_features(ev, meta, session_gap=None):
    """
    Строит признаки по каждому cookie из событий внутри окна.

    Параметры
    ---------
    ev : DataFrame
        События внутри окна (ev_tr или ev_te).
    meta : DataFrame
        train или test, используется для получения полного списка cookie_id.
    session_gap : int, optional
        Если задан, интервалы больше этого порога (в секундах) игнорируются
        при подсчёте статистик по dt — чтобы не смешивать межсессионные паузы.

    Возвращает
    ----------
    DataFrame с колонкой cookie_id и 53 признаками.
    """
    ev = ev.copy()
    ev["event_ts"] = pd.to_datetime(ev["event_ts"])
    ev = ev.sort_values(["cookie_id", "event_ts"])

    # ------------------------------------------------------------------
    # 1. Нормализация платформы
    # ------------------------------------------------------------------
    # Приводим к нижнему регистру и склеиваем синонимы,
    # чтобы WEB/Web/web считались одной платформой.
    ev["platform_clean"] = (
        ev["platform"].astype(str).str.lower().str.strip().replace({"iphone": "ios"})
    )

    # ------------------------------------------------------------------
    # 2. Интервалы между событиями внутри cookie
    # ------------------------------------------------------------------
    # dt — сколько секунд прошло с предыдущего события того же cookie.
    ev["dt"] = ev.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()

    # При необходимости выбрасываем межсессионные паузы.
    if session_gap is not None:
        dt = ev.loc[ev["dt"].notna() & (ev["dt"] <= session_gap), ["cookie_id", "dt"]]
    else:
        dt = ev.loc[ev["dt"].notna(), ["cookie_id", "dt"]]

    # ------------------------------------------------------------------
    # 3. Red-flag паттерны в user_agent
    # ------------------------------------------------------------------
    # UA, которые не могут принадлежать реальному браузеру.
    BOT_STRICT = ["python-urllib", "python-requests", "scrapy", "curl/", "node-fetch"]
    HEADLESS_BOTS = ["headlesschrome"]

    ua_lower = ev["user_agent"].fillna("").str.lower()
    ev["ua_is_bot_strict"] = ua_lower.apply(
        lambda u: int(any(p in u for p in BOT_STRICT))
    )
    ev["ua_is_headless"] = ua_lower.apply(
        lambda u: int(any(p in u for p in HEADLESS_BOTS))
    )
    ev["ua_is_any_bot"] = ((ev["ua_is_bot_strict"] + ev["ua_is_headless"]) > 0).astype(
        int
    )

    # Общие характеристики строки UA.
    ev["ua_length"] = ev["user_agent"].fillna("").str.len()
    ev["ua_n_parts"] = ev["user_agent"].fillna("").str.count(";")

    # ------------------------------------------------------------------
    # 4. Время суток
    # ------------------------------------------------------------------
    ev["hour"] = ev["event_ts"].dt.hour
    ev["is_night"] = (ev["hour"] < 5).astype(int)
    ev["is_morning"] = ((ev["hour"] >= 6) & (ev["hour"] < 12)).astype(int)
    ev["is_evening"] = ((ev["hour"] >= 18) & (ev["hour"] < 24)).astype(int)

    # ------------------------------------------------------------------
    # 5. Характеристики поискового запроса
    # ------------------------------------------------------------------
    ev["query_len"] = ev["search_query"].fillna("").astype(str).str.len()
    ev["query_words"] = ev["search_query"].fillna("").astype(str).str.split().str.len()

    # ------------------------------------------------------------------
    # 6. Группировки
    # ------------------------------------------------------------------
    g_all = ev.groupby("cookie_id")
    g_dt = dt.groupby("cookie_id")["dt"]
    g_sp = g_all["search_page"]

    # ------------------------------------------------------------------
    # 7. Pointer: пропуски координат на web/desktop
    # ------------------------------------------------------------------
    # На мобильных координат нет по определению,
    # поэтому считаем только среди web/desktop.
    is_desktop_web = ev["platform_clean"].isin(["web", "desktop"])
    no_pointer = ev["pointer_x"].isna() | ev["pointer_y"].isna()

    miss_ptr_desktop = (
        ev.assign(_miss=(is_desktop_web & no_pointer))
        .groupby("cookie_id")["_miss"]
        .sum()
    )
    n_desktop = ev.assign(_d=is_desktop_web).groupby("cookie_id")["_d"].sum()
    pointer_missing_share = miss_ptr_desktop / n_desktop.replace(0, np.nan)

    # ------------------------------------------------------------------
    # 8. Pointer: статистики координат
    # ------------------------------------------------------------------
    # Используем только события, где координаты заполнены.
    ptr = ev.dropna(subset=["pointer_x", "pointer_y"]).copy()

    if len(ptr) > 0:
        # Округляем до целых пикселей — иначе шум float раздувает число уникальных точек.
        ptr["px_round"] = ptr["pointer_x"].round(0)
        ptr["py_round"] = ptr["pointer_y"].round(0)

        # Число уникальных положений курсора на cookie.
        ptr_nunique = (
            ptr[["cookie_id", "px_round", "py_round"]]
            .drop_duplicates()
            .groupby("cookie_id")
            .size()
            .rename("ptr_nunique")
        )

        g_ptr = ptr.groupby("cookie_id")
        ptr_feat = pd.DataFrame(
            {
                "ptr_nunique": ptr_nunique,
                "ptr_x_std": g_ptr["pointer_x"].std(),
                "ptr_y_std": g_ptr["pointer_y"].std(),
                "ptr_x_range": g_ptr["pointer_x"].max() - g_ptr["pointer_x"].min(),
                "ptr_y_range": g_ptr["pointer_y"].max() - g_ptr["pointer_y"].min(),
                "n_ptr_events": g_ptr.size(),
            }
        )
        ptr_feat["ptr_density"] = ptr_feat["ptr_nunique"] / ptr_feat["n_ptr_events"]
    else:
        ptr_feat = pd.DataFrame(
            columns=[
                "ptr_nunique",
                "ptr_x_std",
                "ptr_y_std",
                "ptr_x_range",
                "ptr_y_range",
                "n_ptr_events",
                "ptr_density",
            ]
        )

    # ------------------------------------------------------------------
    # 9. Вспомогательная функция: энтропия категорий
    # ------------------------------------------------------------------
    def cat_entropy(s):
        """Энтропия распределения значений. 0 — одна категория, больше — разнообразнее."""
        p = s.dropna().value_counts(normalize=True).values
        if len(p) == 0:
            return 0.0
        return float(-(p * np.log(p)).sum())

    # ------------------------------------------------------------------
    # 10. Основные признаки
    # ------------------------------------------------------------------
    f = pd.DataFrame(
        {
            # --- Базовые ---
            "n_events": g_all.size(),  # сколько всего событий
            "item_nunique": g_all.item_id.nunique(),  # сколько уникальных товаров
            # --- Темп событий ---
            "dt_median": g_dt.median(),  # медианный интервал между событиями, сек
            "dt_cv": g_dt.std()
            / g_dt.mean(),  # коэффициент вариации интервалов (рваность темпа)
            "slow_share": g_dt.apply(lambda s: (s > 300).mean()),  # доля пауз > 5 минут
            "dt_p90": g_dt.quantile(
                0.90
            ),  # 90-й процентиль интервалов(типично высокие интервалы без влияния единичных выбросов)
            # --- Время суток ---
            "n_unique_hours": g_all[
                "hour"
            ].nunique(),  # сколько разных часов суток задействовано
            "night_share": g_all["is_night"].mean(),  # доля событий ночью (0–5)
            "evening_share": g_all["is_evening"].mean(),  # доля событий вечером (18–24)
            "hour_std": g_all["hour"].std(),  # разброс часов захода
            # --- Глубина поиска (search_page) ---
            "depth_mean": g_sp.mean(),  # средний номер страницы поиска
            "depth_nunique": g_sp.apply(
                lambda s: s.dropna().nunique()
            ),  # сколько разных страниц просмотрено
            "depth_p90": g_sp.quantile(
                0.90
            ),  # 90-й процентиль глубины(типично глубокие заходы без одного случайного на 30-ю страницу)
            "depth_max": g_sp.max(),  # максимальная достигнутая страница
            # --- Pointer: пропуски координат на десктопе ---
            "pointer_missing_desktop": miss_ptr_desktop,  # сколько десктопных событий без координат
            "n_desktop_events": n_desktop,  # сколько всего десктопных событий
            "pointer_missing_share": pointer_missing_share,  # доля десктопных событий без координат
            # --- User-Agent: red-flag паттерны ---
            "n_redflag_events": g_all[
                "ua_is_any_bot"
            ].sum(),  # сколько событий с явно ботовым UA
            # --- User-Agent: характеристики строки ---
            "ua_length_mean": g_all["ua_length"].mean(),  # средняя длина строки UA
            "ua_n_parts_mean": g_all["ua_n_parts"].mean(),  # среднее число ';' в UA
            # --- EID (тип события) ---
            "n_eid": g_all["eid"].nunique(),  # сколько разных типов событий
            "eid_mean": g_all["eid"].mean(),  # средний eid (прокси «однообразия»)
            "eid_std": g_all["eid"].std(),  # разброс eid
            "eid_dominance": (
                ev.groupby(["cookie_id", "eid"]).size().groupby("cookie_id").max()
                / g_all.size()
            ),  # доля самого частого типа события
            # --- Платформа ---
            "n_platforms": g_all["platform_clean"].nunique(),  # сколько разных платформ
            "mobile_share": g_all["platform_clean"].apply(
                lambda s: s.isin(["android", "ios"]).mean()
            ),  # доля мобильных событий
            # --- Товары и категории ---
            "item_category_nunique": g_all["item_category"].apply(
                lambda s: s.dropna().nunique()
            ),  # сколько разных категорий
            "item_location_nunique": g_all["item_location"].apply(
                lambda s: s.dropna().nunique()
            ),  # сколько разных локаций
            "seller_type_nunique": g_all["seller_type"].apply(
                lambda s: s.dropna().nunique()
            ),  # сколько разных типов продавцов
            "private_share": g_all["seller_type"].apply(
                lambda s: (s.str.lower() == "private").mean()
            ),  # доля частных продавцов
            "pro_share": g_all["seller_type"].apply(
                lambda s: (s.str.lower() == "pro").mean()
            ),  # доля профессиональных продавцов
            # --- Товары: концентрация внимания ---
            "item_category_dominance": (
                ev.groupby(["cookie_id", "item_category"])
                .size()
                .groupby("cookie_id")
                .max()
                / g_all.size()
            ),  # доля самой частой категории
            "item_category_entropy": g_all["item_category"].apply(
                cat_entropy
            ),  # энтропия распределения по категориям
            "seller_dominance": (
                ev.groupby(["cookie_id", "seller_type"])
                .size()
                .groupby("cookie_id")
                .max()
                / g_all.size()
            ),  # доля самого частого типа продавца
            "location_dominance": (
                ev.groupby(["cookie_id", "item_location"])
                .size()
                .groupby("cookie_id")
                .max()
                / g_all.size()
            ),  # доля самой частой локации
            # --- Поисковые запросы ---
            "has_search": g_all["search_query"]
            .apply(lambda s: s.notna().any())
            .astype(int),  # был ли поиск вообще
            "search_nunique": g_all["search_query"].apply(
                lambda s: s.dropna().nunique()
            ),  # сколько уникальных запросов
            "query_len_mean": g_all["query_len"].mean(),  # средняя длина запроса
            "query_words_mean": g_all[
                "query_words"
            ].mean(),  # среднее число слов в запросе
        }
    )

    # ------------------------------------------------------------------
    # 11. Производные отношения
    # ------------------------------------------------------------------
    f["items_per_event"] = f["item_nunique"] / f["n_events"].replace(
        0, np.nan
    )  # уникальных товаров на событие
    f["categories_per_item"] = f["item_category_nunique"] / f["item_nunique"].replace(
        0, np.nan
    )  # категорий на товар
    f["search_per_event"] = f["search_nunique"] / f["n_events"].replace(
        0, np.nan
    )  # поисков на событие

    # ------------------------------------------------------------------
    # 12. Pointer-статистики присоединяем отдельно
    # ------------------------------------------------------------------
    f = f.join(ptr_feat, how="left")

    # ------------------------------------------------------------------
    # 13. Выравнивание на всех cookie из meta
    # ------------------------------------------------------------------
    f = meta[["cookie_id"]].merge(f.reset_index(), on="cookie_id", how="left")

    # ------------------------------------------------------------------
    # 14. Обработка пропусков
    # ------------------------------------------------------------------
    # Для счётчиков и флагов NaN логично заменить на 0.
    zero_cols = [
        "n_events",
        "item_nunique",
        "n_desktop_events",
        "n_redflag_events",
        "ua_length_mean",
        "ua_n_parts_mean",
        "pointer_missing_desktop",
        "depth_nunique",
        "n_unique_hours",
        "n_platforms",
        "has_ios",
        "has_search",
        "item_category_nunique",
        "item_location_nunique",
        "seller_type_nunique",
        "n_eid",
        "n_ptr_events",
        "ptr_nunique",
    ]
    zero_cols = [c for c in zero_cols if c in f.columns]
    f[zero_cols] = f[zero_cols].fillna(0)

    # Остальные пропуски (dt_median, depth_mean, pointer_missing_share,
    # заполним медианой.

    for col in f.columns:
        if col == "cookie_id":
            continue
        if f[col].isna().any():
            f[col] = f[col].fillna(f[col].median())

    return f


# ---------- Использование ----------
Xtr = basic_features(ev_tr, train)
Xte = basic_features(ev_te, test)
ytr = train.target.values

print(Xtr.shape, Xte.shape)
print("Признаков:", len([c for c in Xtr.columns if c != "cookie_id"]))
Xtr.head()

(11091, 50) (4909, 50)
Признаков: 49


,cookie_id,n_events,item_nunique,dt_median,dt_cv,slow_share,dt_p90,n_unique_hours,night_share,evening_share,...,items_per_event,categories_per_item,search_per_event,ptr_nunique,ptr_x_std,ptr_y_std,ptr_x_range,ptr_y_range,n_ptr_events,ptr_density
0,ck_54a059eb7d3ea68b,7,4,21.0,1.076026,0.000000,59.0,1,1.00000,0.000000,...,0.571429,0.750000,0.428571,0.0,545.491959,305.351196,1603.0,904.5,0.0,1.000000
1,ck_7e4de46eeab82974,41,19,57.5,2.448360,0.150000,3889.7,8,0.00000,0.341463,...,0.463415,0.052632,0.121951,38.0,509.231969,272.953569,1905.0,949.0,38.0,1.000000
2,ck_9320229ef6304522,36,19,22.0,2.212209,0.200000,4513.4,7,0.50000,0.000000,...,0.527778,0.368421,0.277778,35.0,602.124295,312.027426,1869.0,1035.0,35.0,1.000000
3,ck_30ccd25bc1714ed9,21,10,54.5,2.905849,0.100000,877.9,3,0.52381,0.000000,...,0.476190,0.100000,0.142857,0.0,545.491959,305.351196,1603.0,904.5,0.0,1.000000
4,ck_a77c5f05948cdeef,29,16,51.5,2.523129,0.142857,2476.4,5,0.00000,0.000000,...,0.551724,0.250000,0.206897,27.0,450.787816,297.263925,1761.0,1036.0,28.0,0.964286


## Валидация

Тест лежит **позже** трейна по времени, поэтому и валидацию честно делать по времени, а не
случайным сплитом.

Метрику берём из `metric.py`.

Параметры модели я выбирал с помощью библиотеки Optuna.

In [111]:
from sklearn.ensemble import RandomForestClassifier
from metric import precision_at_recall

is_valid = train.window_start_ts.ge('2026-04-17').values

# ВСЕ признаки, кроме идентификатора
cols = [c for c in Xtr.columns if c != 'cookie_id']

print(f'Признаков в модели: {len(cols)}')
print(cols)

model = RandomForestClassifier(
    n_estimators=491,
    max_depth=13,
    min_samples_leaf=17,
    min_samples_split=10,
    max_features=0.3,
    class_weight='balanced',
    random_state=0)

model.fit(Xtr.loc[~is_valid, cols], ytr[~is_valid])
p_va = model.predict_proba(Xtr.loc[is_valid, cols])[:, 1]

print('P@R0.7:', round(precision_at_recall(ytr[is_valid], p_va), 4))
print('доля ботов:', round(ytr[is_valid].mean(), 4))

Признаков в модели: 49
['n_events', 'item_nunique', 'dt_median', 'dt_cv', 'slow_share', 'dt_p90', 'n_unique_hours', 'night_share', 'evening_share', 'hour_std', 'depth_mean', 'depth_nunique', 'depth_p90', 'depth_max', 'pointer_missing_desktop', 'n_desktop_events', 'pointer_missing_share', 'n_redflag_events', 'ua_length_mean', 'ua_n_parts_mean', 'n_eid', 'eid_mean', 'eid_std', 'eid_dominance', 'n_platforms', 'mobile_share', 'item_category_nunique', 'item_location_nunique', 'seller_type_nunique', 'private_share', 'pro_share', 'item_category_dominance', 'item_category_entropy', 'seller_dominance', 'location_dominance', 'has_search', 'search_nunique', 'query_len_mean', 'query_words_mean', 'items_per_event', 'categories_per_item', 'search_per_event', 'ptr_nunique', 'ptr_x_std', 'ptr_y_std', 'ptr_x_range', 'ptr_y_range', 'n_ptr_events', 'ptr_density']
P@R0.7: 0.6829
доля ботов: 0.082


## Сабмит

In [ ]:
model.fit(Xtr[cols], ytr)
sub = pd.DataFrame({
    'cookie_id': Xte.cookie_id,
    'score': model.predict_proba(Xte[cols])[:, 1],
})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
sub.to_csv('submission.csv', index=False)
sub.head()